# "But wait... there's more"

## A More Visible Agent Loop

The Digital Twin contained an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't feel very... loopy.

### Adding 2 more ingredients to make it more real

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls


In [10]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
load_dotenv(override=True)

True

In [11]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [12]:
openai = OpenAI(api_key=os.environ.get('GROQ_API_KEY'),base_url='https://api.groq.com/openai/v1')

In [13]:
# Some lists!

checklist = []
completed = []

In [14]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [15]:
get_checklist_report()

''

In [16]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [17]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [18]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [19]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [20]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [21]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [22]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [23]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

def loop(messages):
    response = openai.chat.completions.create(model="openai/gpt-oss-20b", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="openai/gpt-oss-20b", messages=messages, tools=tools)
    show(response.choices[0].message.content)
    
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
should i go to gym today
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]
checklist, completed = [], []
loop(messages)

**Short answer:**  
If you’re feeling rested, motivated, and your schedule allows it, going to the gym today can boost your fitness and
mood. If you’re sore, tired, or have conflicting commitments, a rest day or a lighter activity (e.g., walking, 
stretching) might be better.

**Things to consider:**

- **Current energy level:** Are you feeling alert or fatigued?  
- **Recovery status:** Any recent heavy training or injury?  
- **Schedule:** Do you have work, family, or other obligations that might interfere?  
- **Goals:** Does today align with your workout plan (e.g., strength, cardio, rest)?  

**Quick decision guide**

| Situation | Action |
|---|---|
| **Feeling energetic & no conflicts** | Hit the gym – keep momentum. |
| **Feeling tired or sore** | Rest or do a low‑intensity session (yoga, light cardio). |
| **Busy with other commitments** | Skip or shorten workout; plan for another day. |

Feel free to adjust based on how you’re feeling today!

NameError: name 'openai' is not defined

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>